## IMPORT AND SETUP

In [ ]:
import os
import sys
import torch
import gc

# Khai báo đường dẫn root để import các module từ thư mục core
sys.path.append(os.path.abspath("../"))

from core.module_2_models import BioModelManager
from core.module_2_extractor import FeatureExtractor

# Giải phóng bộ nhớ GPU trước khi chạy bộ khung trích xuất mới
torch.cuda.empty_cache()
gc.collect()

print(f"[+] CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] Đang sử dụng thiết bị: {torch.cuda.get_device_name(0)}")

## Matrix Configurations

In [ ]:
# 1. Danh sách các file Parquet cần chạy (Hỗ trợ Train, Val, Test gom trong 1 lần cắm máy)
INPUT_FILES_CONFIG = [
    {
        "input_path": "D:/variant_data/train_split.parquet",
        "output_dir": "D:/variant_data/embeddings/train"
    },
    {
        "input_path": "D:/variant_data/val_split.parquet",
        "output_dir": "D:/variant_data/embeddings/val"
    },
    {
        "input_path": "D:/variant_data/test_clinvar_hq.parquet",
        "output_dir": "D:/variant_data/embeddings/test_clinvar"
    }
]

# 2. Không gian ma trận các mô hình mục tiêu cần trích xuất đặc trưng
# Bắt buộc khai báo chuẩn xác model_id trên HuggingFace và phân loại kiến trúc học (mlm / causal)
MODELS_SPACE = [
    {
        "name": "nt_v1_500m",
        "model_id": "InstaDeepAI/nucleotide-transformer-v1-500m-humanref",
        "model_type": "mlm",
        "seq_type": "dna",
        "batch_size": 64
    },
    {
        "name": "nt_v3_650m",
        "model_id": "InstaDeepAI/nucleotide-transformer-v3-650m",
        "model_type": "mlm",
        "seq_type": "dna",
        "batch_size": 32
    },
    {
        "name": "evo2_1b",
        "model_id": "arc-institute/evo2-1b", # Hoặc đường dẫn checkpoint local
        "model_type": "causal",
        "seq_type": "dna",
        "batch_size": 16
    },
    {
        "name": "esm1b_650m",
        "model_id": "facebook/esm1b_t33_650M_UR50S",
        "model_type": "mlm",
        "seq_type": "protein",
        "batch_size": 32
    },
    {
        "name": "esm2_650m",
        "model_id": "facebook/esm2_t33_650M_UR50D",
        "model_type": "mlm",
        "seq_type": "protein",
        "batch_size": 32
    },
    {
        "name": "esmc_600m",
        "model_id": "evolutionaryscale/esmc-open-v0.1", 
        "model_type": "mlm",
        "seq_type": "protein",
        "batch_size": 32
    }
]

## Core Controller Batch Runner

In [ ]:
# Vòng lặp tối thượng: Duyệt qua từng mô hình -> Tải vào VRAM -> Duyệt qua tất cả các file dữ liệu
for model_cfg in MODELS_SPACE:
    print("=" * 80)
    print(f"[MÔ HÌNH HIỆN TẠI]: KHỞI CHẠY TIẾN TRÌNH TRÍCH XUẤT CHO MÔ HÌNH: {model_cfg['name'].upper()}")
    print("=" * 80)
    
    # 1. Khởi tạo Trình quản lý mô hình và đẩy thẳng lên GPU với độ chính xác Mixed Precision (FP16)
    try:
        model_manager = BioModelManager(
            model_id=model_cfg["model_id"],
            model_type=model_cfg["model_type"],
            device="cuda"
        )
        extractor = FeatureExtractor(model_manager=model_manager)
        
    except Exception as e:
        print(f"[LỖI CHIẾN LƯỢC] Không thể nạp mô hình {model_cfg['name']}. Lỗi chi tiết: {e}")
        print("Tự động bỏ qua để chuyển sang mô hình tiếp theo trong danh sách space...")
        continue

    # 2. Duyệt qua từng file dữ liệu đầu vào ứng với mô hình hiện tại
    for data_cfg in INPUT_FILES_CONFIG:
        input_path = data_cfg["input_path"]
        output_dir = data_cfg["output_dir"]
        
        # Đảm bảo thư mục đích an toàn, tạo tự động nếu chưa tồn tại
        os.makedirs(output_dir, exist_ok=True)
        
        # Thiết lập tên file đầu ra theo quy chuẩn phân biệt cấu hình
        output_prefix = os.path.join(output_dir, f"{model_cfg['name']}")
        
        print(f"\n[+] Đang xử lý tập dữ liệu: {os.path.basename(input_path)}")
        print(f"    -> Dữ liệu nguồn: {input_path}")
        print(f"    -> Tiền tố lưu trữ: {output_prefix}_[strategy].pt")
        
        # Kích hoạt Core Pipeline trích xuất (Chạy Double Forward Pass & Pooling & LLR)
        extractor.run_extraction(
            parquet_path=input_path,
            seq_type=model_cfg["seq_type"],
            batch_size=model_cfg["batch_size"],
            output_prefix=output_prefix
        )
        
        # Giải phóng dung lượng đệm GPU sau khi xử lý xong một file dữ liệu để đảm bảo an toàn bộ nhớ
        torch.cuda.empty_cache()
        gc.collect()
        
    # 3. Hủy hoàn toàn thực thể mô hình hiện tại khỏi VRAM trước khi nạp mô hình tiếp theo
    print(f"\n[+] Đang dọn dẹp và giải phóng VRAM cho cấu hình mô hình: {model_cfg['name']}")
    del model_manager
    del extractor
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "#" * 50)
print("[THÀNH CÔNG RỰC RỠ] ĐÃ HOÀN TẤT TRÍCH XUẤT ĐA PHƯƠNG THỨC CHO TẤT CẢ CÁC MÔ HÌNH NỀN TẢNG!")
print("#" * 50)